In [1]:
import os
import torch
import random
import numpy as np
import matplotlib
%matplotlib inline

from exp.exp_long_term_forecasting import Exp_Long_Term_Forecast
"""
ILITOTAL:

TimeXer: {'d_model': 256, 'patch_len': 8, 'dropout': 0.149108300860876, 'lr': 0.0005947492964929737, 
'batch_size': 32, 'd_ff': 128, 'e_layers': 3, 'seq_len': 104}

iTransformer: {'d_model': 256, 'patch_len': 24, 'dropout': 0.10918708798226734, 'lr': 0.00040382104796558713,
'batch_size': 16, 'd_ff': 2048, 'e_layers': 1, 'seq_len': 60}

"""
if __name__ == '__main__':
    for pl in [24]: #6,12,24,36,48,60
                class Configs(object):
                    # general
                    task_name = 'long_term_forecast'  
                    is_training = 1
                    model_id = 'test'
                    model = "iTransformer"               # [iTransformer, TimeXer]
                    # data loader
                    data = 'custom'
                    root_path = './dataset/'
                    data_path = 'national_illness.csv'
                    features = 'M'                     # [M, S, MS]
                    target = 'OT'                      # only for S or MS task
                    target_index = 4
                    freq = 'h'                         # [s, t, h, d, b, w, m, …]
                    checkpoints = './checkpoints/'
                    

                    d_model = 256
                    patch_len = 24
                    dropout = 0.10918708798226734
                    learning_rate =0.00040382104796558713
                    batch_size = 16
                    d_ff = 2048
                    e_layers = 1
                    seq_len = 60


                    scale_method = "standardscaler"
                    difference = False
                    difforder = 'First'
                    seasonal = 26
                    standardize = False 
                    
                    
                    standardize_diff = False #when standardizing after differencing (not used)

                    subtract_last = 0
                    
                    """For CoIN"""

                    per_h_enable = False
                    per_h_cutoff = 3
                    blend_tail_steps = 4
                    input_blend = False
                    blend_mode = None

                    pred_len = pl
                    label_len = 48

                    factor = 3
                    enc_in = 7
                    dec_in = 7
                    c_out = 7
                    des = 'Exp'

                    itr = 1

                    seasonal_patterns = 'Monthly'
                    inverse = True

                    # optimization
                    num_workers = 10

                    train_epochs = 10
                    patience = 3

                    loss = 'MSE'
                    lradj = 'type1'
                    use_amp = False

                    # imputation task
                    mask_rate = 0.25

                    # anomaly detection task
                    anomaly_ratio = 0.25

                    # model define
                    expand = 2                # Mamba
                    d_conv = 4                # Mamba
                    top_k = 5                  # TimesBlock
                    num_kernels = 6            # Inception
                    n_heads = 8
                    d_layers = 1
                    moving_avg = 25

                    distil = True

                    embed = 'timeF'            # [timeF, fixed, learned]
                    activation = 'gelu'
                    output_attention = False
                    channel_independence = 1   # 0: dependent, 1: independent (FreTS)
                    decomp_method = 'moving_avg'  # [moving_avg, dft_decomp]
                    use_norm = 1
                    down_sampling_layers = 0
                    down_sampling_window = 1
                    down_sampling_method = None    # [avg, max, conv]
                    seg_len = 48                   # for SegRNN

                    # GPU
                    use_gpu = True
                    gpu = 0
                    use_multi_gpu = False
                    devices = '0,1,2,3'

                    # de-stationary projector params
                    p_hidden_dims = [128, 128]
                    p_hidden_layers = 2

                    # metrics
                    use_dtw = False

                    # augmentation
                    augmentation_ratio = 0
                    seed = 2
                    jitter = False
                    scaling = False
                    permutation = False
                    randompermutation = False
                    magwarp = False
                    timewarp = False
                    windowslice = False
                    windowwarp = False
                    rotation = False
                    spawner = False
                    dtwwarp = False
                    shapedtwwarp = False
                    wdba = False
                    discdtw = False
                    discsdtw = False
                    extra_tag = ""




                args = Configs()

                # random seed
                fix_seed = 2021
                random.seed(fix_seed)
                torch.manual_seed(fix_seed)
                np.random.seed(fix_seed)

                args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

                if args.use_gpu and args.use_multi_gpu:
                    args.dvices = args.devices.replace(' ', '')
                    device_ids = args.devices.split(',')
                    args.device_ids = [int(id_) for id_ in device_ids]
                    args.gpu = args.device_ids[0]

                Exp = Exp_Long_Term_Forecast

                if args.is_training:
                    for ii in range(args.itr):
                        # setting record of experiments
                        setting = '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(
                            args.model_id,
                            args.model,
                            args.data,
                            args.features,
                            args.seq_len,
                            args.label_len,
                            args.pred_len,
                            args.d_model,
                            args.n_heads,
                            args.e_layers,
                            args.d_layers,
                            args.d_ff,
                            args.factor,
                            args.embed,
                            args.distil,
                            args.des,ii)

                        exp = Exp(args)  # set experiments
        #                 print('>>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>'.format(setting))
                        exp.train(setting)

        #                 print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
                        exp.test(setting)


                        torch.cuda.empty_cache()
                else:
                    ii = 0
                    setting = '{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}'.format(args.model_id,
                                                                                                                args.model,
                                                                                                                args.data,
                                                                                                                args.features,
                                                                                                                args.seq_len,
                                                                                                                args.label_len,
                                                                                                                args.pred_len,
                                                                                                                args.d_model,
                                                                                                                args.n_heads,
                                                                                                                args.e_layers,
                                                                                                                args.d_layers,
                                                                                                                args.d_ff,
                                                                                                                args.factor,
                                                                                                                args.embed,
                                                                                                                args.distil,
                                                                                                                args.des, ii)

                    exp = Exp(args)  # set experiments
                    print('>>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<'.format(setting))
                    exp.test(setting, test=1)
                    torch.cuda.empty_cache()

Use GPU: cuda:0
per_h_enable: False , per_h_cutoff: 3 , input_blend: False , blend_mode: None , blend_tail_steps: 4
standardscaler
train 593
standardscaler
val 74
standardscaler
test 170
Validation loss decreased (inf --> 0.293793).  Saving model ...
Updating learning rate to 0.00040382104796558713
EarlyStopping counter: 1 out of 3
Updating learning rate to 0.00020191052398279357
Validation loss decreased (0.293793 --> 0.269379).  Saving model ...
Updating learning rate to 0.00010095526199139678
EarlyStopping counter: 1 out of 3
Updating learning rate to 5.047763099569839e-05
Validation loss decreased (0.269379 --> 0.258757).  Saving model ...
Updating learning rate to 2.5238815497849196e-05
EarlyStopping counter: 1 out of 3
Updating learning rate to 1.2619407748924598e-05
Validation loss decreased (0.258757 --> 0.255410).  Saving model ...
Updating learning rate to 6.309703874462299e-06
EarlyStopping counter: 1 out of 3
Updating learning rate to 3.1548519372311495e-06
EarlyStopping co